# Task 2 — Technical indicators (TA-Lib + PyNance)

Loads the OHLCV extract, enforces dtypes, handles missing rows, then computes **SMA**, **EMA**, **RSI**, and **MACD** with TA-Lib. PyNance `tech.simple.ret` provides a cross-check for simple returns used in risk-style metrics.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import talib
import pynance.tech.simple as pnts

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 8)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

prices = pd.read_csv(ROOT / "data" / "raw" / "stock_prices_sample.csv")
prices.head()


## Choose a ticker and clean the panel


In [ ]:
SYMBOL = "AAPL"
df = prices.loc[prices["stock"] == SYMBOL].copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").set_index("Date")

num_cols = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

before = len(df)
df = df.dropna(subset=["Adj Close", "Close"])
print(f"Dropped {before - len(df)} rows missing core price fields")

# Basic sanity: no negative prices
assert (df["Adj Close"] > 0).all()

df[num_cols].describe()


## TA-Lib indicators


In [ ]:
close = df["Adj Close"].astype(float)
high = df["High"].astype(float)
low = df["Low"].astype(float)

df["SMA_20"] = talib.SMA(close, timeperiod=20)
df["SMA_50"] = talib.SMA(close, timeperiod=50)
df["EMA_20"] = talib.EMA(close, timeperiod=20)

df["RSI_14"] = talib.RSI(close, timeperiod=14)

macd, macd_signal, macd_hist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
df["MACD"] = macd
df["MACD_signal"] = macd_signal
df["MACD_hist"] = macd_hist

# PyNance simple return (prior session -> today), index aligned to valuation date
pn_returns = pnts.ret(df[["Adj Close"]], selection="Adj Close", n_sessions=1)
df["pn_return"] = pn_returns.reindex(df.index)["Return"]
df["pct_change_close"] = df["Adj Close"].pct_change()
print("Mean abs diff PyNance vs pandas pct_change:", (df["pn_return"] - df["pct_change_close"]).abs().mean())


## Visual diagnostics


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(df.index, df["Adj Close"], label="Adj Close", color="black")
axes[0].plot(df.index, df["SMA_20"], label="SMA 20", color="tab:blue", alpha=0.8)
axes[0].plot(df.index, df["EMA_20"], label="EMA 20", color="tab:orange", alpha=0.8)
axes[0].plot(df.index, df["SMA_50"], label="SMA 50", color="tab:green", alpha=0.8)
axes[0].set_title(f"{SYMBOL} — price vs moving averages")
axes[0].legend(loc="upper left", ncol=2)

axes[1].plot(df.index, df["RSI_14"], color="purple")
axes[1].axhline(70, color="red", ls="--", lw=1)
axes[1].axhline(30, color="green", ls="--", lw=1)
axes[1].set_title("RSI(14)")
axes[1].set_ylabel("RSI")

axes[2].plot(df.index, df["MACD"], label="MACD", color="blue")
axes[2].plot(df.index, df["MACD_signal"], label="Signal", color="red")
axes[2].bar(df.index, df["MACD_hist"], label="Hist", color="gray", alpha=0.4)
axes[2].set_title("MACD(12,26,9)")
axes[2].legend(loc="upper left", ncol=3)

plt.tight_layout()
plt.show()


## Data prep notes (for the interim / final report)

- **Adjusted closes** feed return math and MACD/RSI so corporate actions do not spuriously shift momentum signals.
- **Warm-up rows**: SMA/EMA/MACD need initial windows; drop leading `NaN`s before exporting features to ML.
- **Quality issues spotted**: verify duplicate calendar rows per symbol, split-adjustment consistency, and outliers (bad prints). This sample is synthetic — rehearse the same checks on YFinance pulls.
